# NYC Yellow Taxi Fleet Deployment and Revenue Optimization
### Notebook 02: Predictive modeling and prescriptive deployment analysis

**Course:** DSS142P | **Group 1:** Abad, Aguirre, Galit, Ibañez, Risma, Vicente

**Input:** `data/NYC_Taxi_ABT.parquet`, built by `01_abt.ipynb` (3,757,895 valid trips, Jan–Nov 2025)
**Run order:** `01_abt.ipynb` → this notebook

## Business context

New York City yellow taxi drivers earn only while a passenger is on board, and not every dollar a passenger pays stays with the driver. Each trip carries pass-through charges: the NYS congestion surcharge ($2.50), the MTA tax ($0.50), the improvement surcharge ($1.00), the airport fee ($1.75), and, since **January 5, 2025**, the MTA Congestion Relief Zone fee ($0.75 per yellow-taxi trip in Manhattan's Central Business District). Because these charges are flat per trip, they take a larger share of short, low-fare trips: about 31% of gross revenue on fares under $10, versus about 6% on fares of $40 or more (`01_abt.ipynb`, §6).

How much a driver keeps per occupied hour also depends on **where** and **when** they pick up passengers. Some zones and hours produce longer, better-paying trips, while traffic congestion stretches trip time without raising the fare proportionally. A fleet that positions its drivers well can earn more from the same driving hours.

## Business question and problem

**Business question.** *How can the fleet operator optimize shift scheduling and zone positioning to maximize driver hourly earnings and fleet revenue in the face of 2025 congestion fees?*

**Problem.** The fleet needs to know which pickup zones, days and hours to prioritize for driver deployment to maximize **net revenue per occupied driving hour**, and how congestion fees, trip length, fare level and traffic conditions affect that revenue efficiency.

**Output.** A decision-support model that predicts net revenue per occupied hour for each pickup zone × day × hour, converted into a **ranked deployment table**, a **fee-based fare and trip-length rule**, and **congestion thresholds** for deprioritizing zones.

| Item | Definition |
|---|---|
| Unit of analysis | Pickup zone × date × pickup hour, aggregated from trips |
| Target | `net_revenue_per_hour` = Σ net revenue ÷ Σ occupied minutes × 60, per zone-hour |
| Net revenue | Gross fare total − (CBD fee + congestion surcharge + airport fee + MTA tax + improvement surcharge) − tolls (ABT decision D1) |

## Analytical questions

1. **AQ1: Shift scheduling and zone positioning.** Which day-of-week and pickup-hour combinations should the fleet prioritize for driver deployment based on trip demand and revenue generated per occupied driving hour, and how does this vary across pickup zones?
2. **AQ2: Congestion-fee impact and thresholds.** To what extent do 2025 congestion fees reduce gross trip revenue across different fare tiers and pickup zones, and what trip-length thresholds or minimum fare should be applied to guide drivers to optimal zone-positioning to reduce revenue loss?
3. **AQ3: Congestion tipping points.** At what levels of traffic congestion do pickup zones begin to show lower net take per occupied driving hour, and which zones should the fleet deprioritize when congestion becomes too costly?

| AQ | Method | Output (section) |
|---|---|---|
| AQ1 | Champion model predictions per zone × day × hour, combined with expected demand | Ranked deployment table (§8.1) |
| AQ2 | Grouped analysis of fee share and net $/hr by fare tier, trip-length tier and zone (trip-level ABT) | Minimum-fare / trip-length rule and highest-burden zones (§8.2) |
| AQ3 | Champion model response to lagged traffic speed, overall and per zone | Speed thresholds and zones to deprioritize (§8.3) |

## Methodology

**Approach:** supervised regression + prescriptive ranking, following the comparative structure of Kankanamge et al. (2019), which compares XGBoost with SVR and a neural network on large-scale taxi data.

1. **Data (§2–3).** Load the validated ABT, confirm its structure, and explore the target by hour, day, zone, demand and traffic.
2. **Aggregation and features (§4).** Aggregate trips to zone × date × hour; build lagged demand, speed and fee-share features. **Leakage rule:** only information available before the hour is used as a predictor.
3. **Validation design (§5).** Expanding-window monthly folds (train on all earlier months, validate on the next month) are used for tuning. October–November 2025 is held out as the final test period and used once.
4. **One procedure for every model (§6).** Naive baseline, XGBoost, SVR and neural network all follow: tune on the folds → refit on all pre-test data → evaluate once on the test period. Preprocessing is fit inside the training data only.
5. **Evaluation (§7).** MAE (primary), RMSE and R²; ranking metrics (overall and within-hour Spearman, Precision@10); a day-level bootstrap confidence interval for each model's improvement over the baseline; and errors by segment (peak/off-peak, trip volume).
6. **Champion selection (§7).** Lowest test MAE, provided its improvement over the baseline is supported by the confidence interval; ranking metrics as supporting evidence.
7. **Interpretation and prescription (§8).** SHAP explanations of the champion, then the AQ1–AQ3 outputs.

### Changes from FA2

| FA2 issue | Fix in this version |
|---|---|
| ABT code, ABT file and model inputs were three different versions | Single rebuilt ABT (`01_abt.ipynb`) with documented findings and checks |
| XGBoost refit on train + validation; other models were not | One procedure for all models, including the baseline |
| Single validation split; MAE rose from validation to test | Expanding-window monthly folds |
| Spearman reported for XGBoost only; no ranking metric for dispatch | Spearman, within-hour Spearman and Precision@10 for every model |
| No uncertainty on the gain over the baseline | Day-level bootstrap confidence intervals |
| Feature importance presented as recommendations | Ranked deployment table, fee rule and congestion thresholds |
| Errors reported only as overall averages | Segment-level error analysis |
| SVR trained on a 25,000-row subsample (47 min) | Kernel-approximated SVR on the full training data |

## 1. Setup

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

notebook_start = time.perf_counter()

# ---- Paths ----
DATA_DIR = Path("../data")
ABT_PATH = DATA_DIR / "NYC_Taxi_ABT.parquet"
ZONES_PATH = DATA_DIR / "taxi_zone_lookup.csv"
FIG_DIR = Path("../reports/figures")
TABLE_DIR = Path("../reports/tables")
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ---- Reproducibility ----
RANDOM_STATE = 42

# ---- Validation design (see Methodology) ----
TEST_START = pd.Timestamp("2025-10-01")   # Oct–Nov 2025: final test period, used once
VALIDATION_MONTHS = [6, 7, 8, 9]          # each fold trains on all earlier months, validates on this one

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

def save_fig(fig, name):
    """Save a figure to reports/figures for use in the write-up."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

for p in [ABT_PATH, ZONES_PATH, FIG_DIR, TABLE_DIR]:
    print(f"{'✓' if p.exists() else '✗ MISSING'}  {p}")

Matplotlib is building the font cache; this may take a moment.


✓  ../data/NYC_Taxi_ABT.parquet
✓  ../data/taxi_zone_lookup.csv
✓  ../reports/figures
✓  ../reports/tables


## 2. Load and validate the ABT
The ABT was validated in `01_abt.ipynb`; these checks confirm the file loaded here is that same, complete version.

In [2]:
abt = pd.read_parquet(ABT_PATH)
print(f"ABT: {len(abt):,} rows × {abt.shape[1]} columns | "
      f"{abt.memory_usage(deep=True).sum() / 1024**3:.2f} GB in memory")

EXPECTED_ROWS = 3_757_895  # output of 01_abt.ipynb §8
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
required = ["tpep_pickup_datetime", "PULocationID", "pickup_hour", "day_of_week", "is_weekend",
            "duration_min", "speed_mph", "trip_distance", "fare_amount", "gross_revenue",
            "total_fees", "congestion_fees", "tolls_amount", "tip_amount", "net_revenue",
            "net_revenue_ex_tip", "net_revenue_per_hour", "fee_impact_pct",
            "congestion_fee_pct", "pickup_date", "pickup_month", "fare_tier", "trip_length_tier"]

recomputed_target = abt["net_revenue"] / abt["duration_min"] * 60
is_cat = lambda col: isinstance(abt[col].dtype, pd.CategoricalDtype)

checks = {
    "Row count matches 01_abt output":     len(abt) == EXPECTED_ROWS,
    "All required columns present":        set(required).issubset(abt.columns),
    "No missing values":                   abt.isna().sum().sum() == 0,
    "Pickups within Jan–Nov 2025":         abt["tpep_pickup_datetime"].between("2025-01-01", "2025-12-01").all(),
    "Target matches its formula":          np.allclose(recomputed_target, abt["net_revenue_per_hour"]),
    "Day order kept (Mon → Sun)":          is_cat("day_of_week") and list(abt["day_of_week"].cat.categories) == DAY_ORDER,
    "Tiers kept as ordered categories":    is_cat("fare_tier") and is_cat("trip_length_tier"),
}
results = pd.Series(checks, name="passed")
display(results.to_frame())
assert results.all(), f"Failed: {list(results[~results].index)}"
print("ABT validated ✓")

ABT: 3,757,895 rows × 37 columns | 0.73 GB in memory


,passed
Row count matches 01_abt output,True
All required columns present,True
No missing values,True
Pickups within Jan–Nov 2025,True
Target matches its formula,True
Day order kept (Mon → Sun),True
Tiers kept as ordered categories,True


ABT validated ✓


In [3]:
monthly = (abt.groupby("pickup_month")
              .agg(trips=("net_revenue", "size"),
                   avg_net_per_hour=("net_revenue_per_hour", "mean"))
              .round(2))
monthly["role"] = np.where(monthly.index >= TEST_START.month, "TEST (held out)",
                  np.where(monthly.index.isin(VALIDATION_MONTHS), "validation fold", "training only"))
monthly

,trips,avg_net_per_hour,role
pickup_month,,,
1,308589,95.87,training only
2,315088,93.20,training only
3,364210,94.57,training only
4,348376,92.34,training only
5,384718,90.09,training only
6,365964,92.41,validation fold
7,328019,91.57,validation fold
8,292101,91.55,validation fold
9,358195,88.48,validation fold


In [4]:
zones = pd.read_csv(ZONES_PATH)
display(zones.head())

abt_zones = abt["PULocationID"].unique()
unnamed = set(abt_zones) - set(zones["LocationID"])
print(f"Pickup zones in ABT: {len(abt_zones)} | zones without a name: {len(unnamed)}")
print("\nPickup zones by borough:")
print(zones[zones["LocationID"].isin(abt_zones)]["Borough"].value_counts().to_string())

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


Pickup zones in ABT: 258 | zones without a name: 0

Pickup zones by borough:
Borough
Queens           69
Manhattan        67
Brooklyn         61
Bronx            43
Staten Island    17
EWR               1
